# Multi-Agent Incident War Room — GRPO training notebook

This notebook trains Qwen2.5 (1.5B on the free T4, 7B on a paid L40S/A100) with GRPO on the War Room environment.

**What you get if you run the whole notebook:**

- A trained LoRA adapter you can push to the Hugging Face Hub
- Per-episode metrics.json and a training curves PNG
- A head-to-head head-to-head comparison against the base model if you run Cell 8

**Time and cost:**

- Smoke run on Colab T4 free: ~10 minutes, 5 episodes on Qwen 1.5B — pipeline validation only
- Headline run on L40S (`hf jobs`): ~25 minutes, 100 episodes on Qwen 7B — the one in the README

Live demo + repo: https://huggingface.co/spaces/brodie1of1/war-room

## 1. Clone the repo + install

In [ ]:
!git clone https://github.com/Git4Lokesh/Meta_Hackathon_ClaudeStalkers.git
%cd Meta_Hackathon_ClaudeStalkers

# Pinned to the TRL window we've tested against. trl>=0.19 uses torch 2.5 APIs
# (torch.distributed.fsdp.fully_shard) that the Colab T4 image doesn't have.
!pip install -q \
    'trl>=0.15.0,<0.19' 'peft>=0.14.0' 'transformers>=4.46.0,<4.50' \
    datasets accelerate bitsandbytes matplotlib
!pip install -q fastapi pydantic uvicorn rich
!pip install -q -e .

## 2. Sanity check — is the environment installed?

This should print 172 tests passing in about a second, and then import the environment to make sure every module is available.

In [ ]:
!PYTHONPATH=. python -m pytest tests/ -q 2>&1 | tail -10

import sys
sys.path.insert(0, '.')
from round2.war_room.environment import WarRoomEnvironment
env = WarRoomEnvironment()
obs = env.reset(task_id='task1', seed=42)
print(f'Env OK. Task: {obs.metadata["task_id"]} · max_rounds={obs.metadata["max_rounds"]}')

## 3. Oracle audit — are all the tasks reachable?

Before spending GPU time on training, confirm the tasks are solvable in principle. A hand-crafted perfect-knowledge policy should score 0.85+. If any task is below that, its verifier is broken and RL can't fix it. We caught two unreachable milestones this way.

In [ ]:
!PYTHONPATH=. python scripts/oracle_audit.py

## 4. Gradient check — does a good completion score higher than a garbage one?

If garbage and correct completions score the same, GRPO has no group-relative advantage and the gradient dies. This script verifies the gradient is alive across every fault type we train on.

In [ ]:
!PYTHONPATH=. python scripts/verify_multirole_gradient.py 2>&1 | tail -30

## 5. Smoke training run

5 episodes × 3 procedural difficulties = 15 prompts. Enough to confirm the pipeline produces a non-zero reward curve. Takes a few minutes on T4 free tier.

**If you're on a paid GPU and want the real run, skip to Cell 6 instead.**

In [ ]:
# Smoke run — Qwen 1.5B fits on T4. Use --no-unsloth if Unsloth misbehaves on your runtime.
!PYTHONPATH=. python round2/war_room/train_colab.py \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 5 \
    --tasks procedural_easy \
    --lenient-format \
    --output outputs/war_room_grpo_smoke \
    --lora-r 16 --lr 5e-6 --generations 4 --batch-size 1

## 6. Headline training run (paid GPU)

This is the configuration that produced the v3 adapter in the README: 100 episodes × 3 procedural difficulties = 300 gradient updates, rank-16 LoRA, about 25 minutes of L40S time.

Skip this cell if you're on a free T4. Use Cell 5 instead and trust the v3 adapter we've already published at `brodie1of1/war-room-grpo-adapter-v3`.

In [ ]:
!PYTHONPATH=. python round2/war_room/train_colab.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --episodes 100 \
    --tasks procedural_easy procedural_medium procedural_hard \
    --lenient-format --no-unsloth \
    --output outputs/war_room_grpo_v3 \
    --lora-r 16 --lr 5e-6 --generations 4 --batch-size 1

## 7. Look at the training curves

The milestone reward is the one that matters. Format and anti-hack saturate at 1.0 from step 1 because Qwen already emits structured output and never loops.

In [ ]:
import json
import matplotlib.pyplot as plt

# Pick whichever run you did in Cell 5 or 6
run_dir = 'outputs/war_room_grpo_v3'  # or 'outputs/war_room_grpo_smoke'
with open(f'{run_dir}/metrics.json') as f:
    metrics = json.load(f)

rewards = metrics['team_reward']
milestones = metrics.get('milestones_achieved', [0] * len(rewards))
rounds = metrics.get('rounds_used', [0] * len(rewards))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(rewards, alpha=0.3, label='per-episode')
window = max(1, len(rewards) // 20)
smoothed = [sum(rewards[max(0,i-window):i+1])/(min(i+1, window)) for i in range(len(rewards))]
axes[0].plot(smoothed, linewidth=2, label=f'rolling mean (window={window})')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('team_reward')
axes[0].set_title('Team reward over training')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(milestones, alpha=0.5)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('milestones_hit')
axes[1].set_title('Milestones achieved per episode')
axes[1].grid(alpha=0.3)

axes[2].plot(rounds, alpha=0.5)
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('rounds')
axes[2].set_title('Episode length')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{run_dir}/training_curves_notebook.png', dpi=100)
plt.show()

print(f'Mean team_reward: {sum(rewards)/len(rewards):.3f}')
print(f'Max team_reward:  {max(rewards):.3f}')
print(f'First-20 mean: {sum(rewards[:20])/20:.3f}')
print(f'Last-20 mean:  {sum(rewards[-20:])/20:.3f}')

## 8. Head-to-head eval: base model vs your trained adapter

Run both models through the three scripted tasks with 5 seeds each and compare. Runs locally if you have GPU memory; otherwise use `hf_job_llm_eval.sh` to run it on HF Jobs.

**Requires a GPU with enough memory for Qwen 7B in bf16 (~14 GB).**

In [ ]:
# Quick eval — 3 tasks × 5 seeds × 2 models = 30 rollouts
!PYTHONPATH=. python round2/war_room/eval_llm_on_gpu.py \
    --seeds 11 22 33 44 55 \
    --tasks task1 task2 task3 \
    --adapter-repo brodie1of1/war-room-grpo-adapter-v3 \
    --output-dir outputs/llm_eval/notebook

# Read the summary
import json
with open('outputs/llm_eval/notebook/summary.json') as f:
    summary = json.load(f)

print('\n=== Head-to-head ===')
for model, tasks in summary['per_model_task'].items():
    print(f'\n{model}:')
    for task, metrics in tasks.items():
        print(f'  {task}: avg_score={metrics["avg_score"]:.3f}')

print(f'\nComposite delta (trained − base): {summary["delta_composite"]:+.3f}')

## 9. Push the adapter to the Hub (optional)

Only runs if you've logged in with a token that has write access.

In [ ]:
# Uncomment to push. Replace the repo name with your own account.
# from huggingface_hub import HfApi, create_repo
# import os
# repo = 'YOUR_HF_USERNAME/war-room-grpo-adapter'
# api = HfApi()
# create_repo(repo, exist_ok=True)
# api.upload_folder(folder_path='outputs/war_room_grpo_v3', repo_id=repo, repo_type='model')
# print(f'Pushed to https://huggingface.co/{repo}')